# rustree Python API

This notebook documents the full Python API for **rustree**, a Rust library for simulating and analyzing phylogenetic trees.

**Contents:**
1. [Birth-Death Species Tree Simulation](#1-birth-death-species-tree-simulation)
2. [DTL Gene Tree Simulation](#2-dtl-gene-tree-simulation)

---

## 1. Birth-Death Species Tree Simulation

The birth-death (BD) model simulates species trees where lineages split (speciation, rate `lambda`) and go extinct (extinction, rate `mu`). The simulation conditions on producing exactly `n` extant species.

### 1.1 `rustree.simulate_species_tree(n, lambda_, mu, seed=None)`

Simulate a species tree under the birth-death process.

**Parameters:**
- `n` (int) — Number of extant species. Must be > 0.
- `lambda_` (float) — Speciation (birth) rate. Must be > 0.
- `mu` (float) — Extinction (death) rate. Must be >= 0 and strictly less than `lambda_`.
- `seed` (int, optional) — Random seed for reproducibility.

**Returns:** `PySpeciesTree`

In [ ]:
import rustree

# Simulate a species tree with 10 extant species
species_tree = rustree.simulate_species_tree(n=10, lambda_=1.0, mu=0.5, seed=42)
print(f"Nodes: {species_tree.num_nodes()}")
print(f"Leaves: {species_tree.num_leaves()}")
print(f"Height: {species_tree.tree_height():.4f}")
print(f"Newick: {species_tree.to_newick()}")

### 1.2 `rustree.parse_species_tree(newick_str)`

Parse a species tree from a Newick string or file path.

**Parameters:**
- `newick_str` (str) — A Newick-formatted string (e.g. `"((A:1,B:1):1,C:2):0;"`) or a path to a `.nwk` file.

**Returns:** `PySpeciesTree`

In [ ]:
# From a Newick string
tree = rustree.parse_species_tree("((A:1,B:1):1,C:2):0;")
print(f"Leaves: {tree.leaf_names()}")
print(f"Nodes: {tree.num_nodes()}, Height: {tree.tree_height():.1f}")

# From a file (uncomment if you have a file)
# tree = rustree.parse_species_tree("path/to/species.nwk")

### 1.3 `PySpeciesTree` methods

| Method | Returns | Description |
|--------|---------|-------------|
| `to_newick()` | `str` | Newick string representation |
| `num_nodes()` | `int` | Total number of nodes (internal + leaves) |
| `num_leaves()` | `int` | Number of extant species (leaf nodes) |
| `tree_height()` | `float` | Depth of the deepest leaf |
| `root_index()` | `int` | Index of the root node |
| `leaf_names()` | `list[str]` | Names of all leaf nodes |
| `save_newick(filepath)` | `None` | Save tree to a Newick file |
| `to_svg(filepath=None, open_browser=False, internal_names=True)` | `str` | Generate SVG visualization (requires `thirdkind`) |
| `display(internal_names=True)` | SVG | Display in Jupyter notebook |
| `extract_induced_subtree_by_names(names)` | `PySpeciesTree` | Extract subtree keeping only specified leaves |
| `save_bd_events_csv(filepath)` | `None` | Save birth-death events to CSV |
| `get_bd_events()` | `dict` | Get events as dict (suitable for `pd.DataFrame`) |
| `get_ltt_data()` | `dict` | Get Lineages Through Time data |
| `plot_ltt(filepath=None, title=None, xlabel=None, ylabel=None)` | `None` | Plot LTT using matplotlib |
| `pairwise_distances(distance_type, leaves_only=True)` | `DataFrame` | Pairwise distances between nodes |
| `save_pairwise_distances_csv(filepath, distance_type, leaves_only=True)` | `None` | Save distances to CSV |

#### Visualization

In [ ]:
species_tree = rustree.simulate_species_tree(n=8, lambda_=1.0, mu=0.3, seed=42)
species_tree.display()

In [ ]:
# Save to file
species_tree.to_svg("species_tree.svg")

# Open in browser
# species_tree.to_svg(open_browser=True)

# Hide internal node names
species_tree.display(internal_names=False)

#### Induced subtree extraction

In [ ]:
species_tree = rustree.simulate_species_tree(n=10, lambda_=1.0, mu=0.3, seed=42)
all_leaves = species_tree.leaf_names()
print(f"Original: {species_tree.num_leaves()} leaves — {all_leaves}")

# Keep only a subset of species
subset = all_leaves[:4]
subtree = species_tree.extract_induced_subtree_by_names(subset)
print(f"Subtree: {subtree.num_leaves()} leaves — {subtree.leaf_names()}")
print(f"Subtree Newick: {subtree.to_newick()}")

#### Birth-death events

In [ ]:
import pandas as pd

species_tree = rustree.simulate_species_tree(n=6, lambda_=1.0, mu=0.3, seed=42)

# Get events as a dictionary, convert to DataFrame
events = species_tree.get_bd_events()
df = pd.DataFrame(events)
df

In [ ]:
# Save directly to CSV
species_tree.save_bd_events_csv("bd_events.csv")
print("Saved to bd_events.csv")

#### Lineages Through Time (LTT)

In [ ]:
import matplotlib.pyplot as plt

species_tree = rustree.simulate_species_tree(n=50, lambda_=1.0, mu=0.5, seed=42)

# Option 1: get raw data and plot manually
ltt = species_tree.get_ltt_data()
plt.step(ltt['times'], ltt['lineages'], where='post')
plt.xlabel('Time (past to present)')
plt.ylabel('Number of lineages')
plt.title('Lineages Through Time')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Option 2: use the built-in plot method
species_tree.plot_ltt(title="LTT for 50-species tree")

# Save to file
# species_tree.plot_ltt(filepath="ltt.png")

#### Pairwise distances

In [ ]:
species_tree = rustree.simulate_species_tree(n=5, lambda_=1.0, mu=0.3, seed=42)

# Metric distances (sum of branch lengths) between leaves
df_metric = species_tree.pairwise_distances("metric", leaves_only=True)
print("Metric (patristic) distances:")
display(df_metric)

# Topological distances (number of edges) between all nodes
df_topo = species_tree.pairwise_distances("topological", leaves_only=False)
print(f"\nTopological distances (all {len(df_topo)} pairs):")
display(df_topo.head(10))

In [ ]:
# Save to CSV
species_tree.save_pairwise_distances_csv("distances.csv", "metric", leaves_only=True)
print("Saved to distances.csv")

#### Saving trees

In [ ]:
species_tree = rustree.simulate_species_tree(n=10, lambda_=1.0, mu=0.5, seed=42)

# Save Newick
species_tree.save_newick("species.nwk")

# Get Newick string
nwk = species_tree.to_newick()
print(nwk)

---

## 2. DTL Gene Tree Simulation

The DTL model simulates gene trees along a species tree, with three types of events:
- **Duplication** (rate `lambda_d`): a gene copy is created within the same species
- **Transfer** (rate `lambda_t`): a gene copy moves to a contemporaneous species
- **Loss** (rate `lambda_l`): a gene copy is lost

Two simulation models are available:
- **Per-gene-copy** (`simulate_dtl`): event rate scales with the number of gene copies. More gene copies = more events.
- **Per-species** (`simulate_dtl_per_species`): event rate scales with the number of species hosting genes. This is the model used by Zombi and is appropriate when DTL events are driven by species-level processes.

### 2.1 `species_tree.simulate_dtl(...)`

Simulate a single gene tree using the **per-gene-copy** model.

**Parameters:**
- `lambda_d` (float) — Duplication rate per gene copy per unit time. Must be >= 0.
- `lambda_t` (float) — Transfer rate per gene copy per unit time. Must be >= 0.
- `lambda_l` (float) — Loss rate per gene copy per unit time. Must be >= 0.
- `transfer_alpha` (float, optional) — Distance decay parameter for **assortative transfers**. Higher values make transfers to closely related species more likely. `None` = uniform random recipient.
- `replacement_transfer` (float, optional) — Probability that a transfer **replaces** a gene in the recipient species (causing a loss in the recipient). Must be in [0, 1]. `None` or `0.0` = no replacement.
- `require_extant` (bool) — If `True`, retry simulation until at least one gene survives to the present. Default: `False`.
- `seed` (int, optional) — Random seed for reproducibility.

**Returns:** `PyGeneTree`

In [ ]:
import rustree

species_tree = rustree.simulate_species_tree(n=10, lambda_=1.0, mu=0.3, seed=42)

# Basic DTL simulation
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.5,
    lambda_t=0.2,
    lambda_l=0.3,
    seed=123
)

s, d, t, l, leaves = gene_tree.count_events()
print(f"Events: S={s}, D={d}, T={t}, L={l}, Leaves={leaves}")
print(f"Nodes: {gene_tree.num_nodes()}, Extant: {gene_tree.num_extant()}")

### 2.2 `species_tree.simulate_dtl_batch(n, ...)`

Simulate multiple gene trees efficiently. Faster than calling `simulate_dtl` in a loop because species tree data (depths, LCA table) is precomputed once.

**Parameters:** Same as `simulate_dtl`, plus:
- `n` (int) — Number of gene trees to simulate.

**Returns:** `list[PyGeneTree]`

In [ ]:
import time

species_tree = rustree.simulate_species_tree(n=20, lambda_=1.0, mu=0.3, seed=42)

start = time.time()
gene_trees = species_tree.simulate_dtl_batch(
    n=1000,
    lambda_d=0.1,
    lambda_t=0.1,
    lambda_l=0.05,
    seed=123
)
elapsed = time.time() - start
print(f"Simulated {len(gene_trees)} gene trees in {elapsed:.3f}s")

# Summary statistics
extant_counts = [gt.num_extant() for gt in gene_trees]
print(f"Mean extant genes: {sum(extant_counts)/len(extant_counts):.1f}")
print(f"Min: {min(extant_counts)}, Max: {max(extant_counts)}")

### 2.3 `species_tree.simulate_dtl_per_species(...)` and `simulate_dtl_per_species_batch(n, ...)`

Simulate gene trees using the **Zombi-style per-species** model. The total event rate is proportional to the number of *species* with gene copies, not the total number of gene copies.

When an event occurs:
1. A random species (among those with genes) is chosen
2. A random gene copy within that species is affected

This means duplications don't increase the event rate, producing more conservative gene tree sizes.

**Parameters:** Same as `simulate_dtl` / `simulate_dtl_batch`.

**Returns:** `PyGeneTree` / `list[PyGeneTree]`

In [ ]:
species_tree = rustree.simulate_species_tree(n=15, lambda_=1.0, mu=0.3, seed=42)

# Compare per-gene-copy vs per-species
gt_copy = species_tree.simulate_dtl(
    lambda_d=0.5, lambda_t=0.2, lambda_l=0.3, seed=123
)
gt_species = species_tree.simulate_dtl_per_species(
    lambda_d=0.5, lambda_t=0.2, lambda_l=0.3, seed=123
)

print(f"Per-gene-copy model: {gt_copy.num_extant()} extant genes, {gt_copy.num_nodes()} nodes")
print(f"Per-species model:   {gt_species.num_extant()} extant genes, {gt_species.num_nodes()} nodes")

### 2.4 Advanced options: assortative transfers and replacement transfers

In [ ]:
species_tree = rustree.simulate_species_tree(n=15, lambda_=1.0, mu=0.3, seed=42)

# Assortative transfer: higher alpha = stronger preference for closely related recipients
gt_uniform = species_tree.simulate_dtl(
    lambda_d=0.1, lambda_t=1.0, lambda_l=0.1,
    transfer_alpha=None,  # uniform random
    seed=456
)
gt_assortative = species_tree.simulate_dtl(
    lambda_d=0.1, lambda_t=1.0, lambda_l=0.1,
    transfer_alpha=5.0,   # prefer close relatives
    seed=456
)

s1, d1, t1, l1, lv1 = gt_uniform.count_events()
s2, d2, t2, l2, lv2 = gt_assortative.count_events()
print(f"Uniform transfers:     T={t1}, extant={gt_uniform.num_extant()}")
print(f"Assortative (alpha=5): T={t2}, extant={gt_assortative.num_extant()}")

In [ ]:
# Replacement transfer: with probability p, the transferred gene replaces
# an existing gene in the recipient species (causing a loss in the recipient)
gt_no_replace = species_tree.simulate_dtl(
    lambda_d=0.1, lambda_t=1.0, lambda_l=0.1,
    replacement_transfer=None,  # no replacement
    seed=789
)
gt_replace = species_tree.simulate_dtl(
    lambda_d=0.1, lambda_t=1.0, lambda_l=0.1,
    replacement_transfer=0.5,   # 50% replacement probability
    seed=789
)

s1, d1, t1, l1, _ = gt_no_replace.count_events()
s2, d2, t2, l2, _ = gt_replace.count_events()
print(f"No replacement:        T={t1}, L={l1}, extant={gt_no_replace.num_extant()}")
print(f"50% replacement:       T={t2}, L={l2}, extant={gt_replace.num_extant()}")

In [ ]:
# require_extant: guarantee at least one surviving gene
# (useful with high loss rates that often produce empty trees)
gt = species_tree.simulate_dtl(
    lambda_d=0.0, lambda_t=0.0, lambda_l=5.0,
    require_extant=True,
    seed=999
)
print(f"High loss + require_extant: {gt.num_extant()} extant genes")

### 2.5 `PyGeneTree` methods

| Method | Returns | Description |
|--------|---------|-------------|
| `to_newick()` | `str` | Gene tree in Newick format |
| `save_newick(filepath)` | `None` | Save gene tree to Newick file |
| `num_nodes()` | `int` | Total nodes (internal + leaves + losses) |
| `num_extant()` | `int` | Number of extant genes (surviving leaves) |
| `count_events()` | `(S, D, T, L, Leaves)` | Count of each event type |
| `extant_gene_names()` | `list[str]` | Names of surviving genes |
| `to_xml()` | `str` | RecPhyloXML string |
| `save_xml(filepath)` | `None` | Save to RecPhyloXML file |
| `to_svg(filepath=None, open_browser=False)` | `str` | SVG visualization (requires `thirdkind`) |
| `display()` | SVG | Display in Jupyter notebook |
| `to_csv(filepath=None)` | `DataFrame` | Gene tree data with columns: node_id, name, parent, left_child, left_child_name, right_child, right_child_name, length, depth, species_node, species_node_left, species_node_right, event |
| `sample_extant()` | `PyGeneTree` | Extract induced subtree of extant genes only |
| `sample_by_names(names)` | `PyGeneTree` | Extract induced subtree keeping only specified genes |
| `sample_species_leaves(species_leaf_names)` | `PyGeneTree` | Sample species and filter gene tree accordingly |
| `pairwise_distances(distance_type, leaves_only=True)` | `DataFrame` | Pairwise distances |
| `save_pairwise_distances_csv(filepath, distance_type, leaves_only=True)` | `None` | Save distances to CSV |

#### Inspecting gene trees

In [ ]:
species_tree = rustree.simulate_species_tree(n=8, lambda_=1.0, mu=0.3, seed=42)
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.3, lambda_t=0.2, lambda_l=0.1, seed=100
)

# Event counts
s, d, t, l, leaves = gene_tree.count_events()
print(f"Speciations: {s}")
print(f"Duplications: {d}")
print(f"Transfers: {t}")
print(f"Losses: {l}")
print(f"Leaves: {leaves}")
print(f"Extant genes: {gene_tree.num_extant()}")
print(f"Extant gene names: {gene_tree.extant_gene_names()}")

#### Visualization

In [ ]:
# Display in notebook (shows gene tree reconciled within species tree)
gene_tree.display()

In [ ]:
# Save to SVG file
gene_tree.to_svg("reconciled_tree.svg")

# Open in browser
# gene_tree.to_svg(open_browser=True)

#### Exporting data

In [ ]:
# Get gene tree as a DataFrame
df = gene_tree.to_csv()
df

In [ ]:
# Save to CSV file and get DataFrame
df = gene_tree.to_csv("gene_tree.csv")
print("Saved to gene_tree.csv")

In [ ]:
# Export as RecPhyloXML
xml = gene_tree.to_xml()
print(xml[:500])  # first 500 chars

# Save to file
gene_tree.save_xml("reconciled.xml")

In [ ]:
# Newick
print(gene_tree.to_newick())
gene_tree.save_newick("gene_tree.nwk")

#### Sampling and pruning gene trees

Three sampling methods are available:
- `sample_extant()` — Keep only surviving genes (remove losses)
- `sample_by_names(names)` — Keep only specified genes
- `sample_species_leaves(species_names)` — Keep only genes mapping to specified species, and prune the species tree accordingly

In [ ]:
species_tree = rustree.simulate_species_tree(n=8, lambda_=1.0, mu=0.3, seed=42)
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.3, lambda_t=0.2, lambda_l=0.5, seed=100
)

s, d, t, l, leaves = gene_tree.count_events()
print(f"Full tree: {gene_tree.num_nodes()} nodes, {gene_tree.num_extant()} extant, {l} losses")

# Remove loss lineages — keep only the observable (extant) gene tree
sampled = gene_tree.sample_extant()
s2, d2, t2, l2, leaves2 = sampled.count_events()
print(f"Sampled:   {sampled.num_nodes()} nodes, {sampled.num_extant()} extant, {l2} losses")

In [ ]:
# Keep only specific genes
extant_names = gene_tree.extant_gene_names()
print(f"All extant: {extant_names}")

if len(extant_names) >= 3:
    subset = extant_names[:3]
    sampled = gene_tree.sample_by_names(subset)
    print(f"Kept {subset} -> {sampled.num_nodes()} nodes")

In [ ]:
# Sample species: keep only genes from selected species,
# and prune the species tree to match
species_tree = rustree.simulate_species_tree(n=8, lambda_=1.0, mu=0.3, seed=42)
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.2, lambda_t=0.1, lambda_l=0.1,
    require_extant=True, seed=200
)

all_species = species_tree.leaf_names()
print(f"All species: {all_species}")

# Keep only first 4 species
selected = all_species[:4]
sampled_gt = gene_tree.sample_species_leaves(selected)
print(f"Sampled gene tree: {sampled_gt.num_nodes()} nodes, {sampled_gt.num_extant()} extant")

#### Pairwise distances in gene trees

In [ ]:
species_tree = rustree.simulate_species_tree(n=5, lambda_=1.0, mu=0.3, seed=42)
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.1, lambda_t=0.1, lambda_l=0.0, seed=123
)

# Metric distances between extant gene leaves
df = gene_tree.pairwise_distances("metric", leaves_only=True)
display(df)

# Save to file
gene_tree.save_pairwise_distances_csv("gene_distances.csv", "metric")

### 2.6 `rustree.parse_recphyloxml(filepath)`

Parse a RecPhyloXML file (e.g., from ALERax or other reconciliation tools) into a `PyGeneTree`.

**Parameters:**
- `filepath` (str) — Path to the RecPhyloXML file.

**Returns:** `PyGeneTree`

In [ ]:
# Round-trip: simulate -> save XML -> parse back
species_tree = rustree.simulate_species_tree(n=5, lambda_=1.0, mu=0.3, seed=42)
gene_tree = species_tree.simulate_dtl(
    lambda_d=0.2, lambda_t=0.1, lambda_l=0.1, seed=123
)

# Save
gene_tree.save_xml("roundtrip_test.xml")

# Parse back
parsed = rustree.parse_recphyloxml("roundtrip_test.xml")
print(f"Original: {gene_tree.count_events()}")
print(f"Parsed:   {parsed.count_events()}")

### 2.7 Batch analysis example

Simulate many gene trees and analyze event distributions.

In [ ]:
import matplotlib.pyplot as plt
import time

species_tree = rustree.simulate_species_tree(n=20, lambda_=1.0, mu=0.3, seed=42)

start = time.time()
gene_trees = species_tree.simulate_dtl_batch(
    n=1000,
    lambda_d=0.2,
    lambda_t=0.1,
    lambda_l=0.1,
    seed=42
)
elapsed = time.time() - start
print(f"Simulated {len(gene_trees)} gene trees in {elapsed:.3f}s")

# Collect statistics
n_leaves = []
n_dup = []
n_transfer = []
n_loss = []

for gt in gene_trees:
    s, d, t, l, leaves = gt.count_events()
    n_leaves.append(gt.num_extant())
    n_dup.append(d)
    n_transfer.append(t)
    n_loss.append(l)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].hist(n_leaves, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Extant genes')
axes[0].set_title(f'Gene family size (mean={sum(n_leaves)/len(n_leaves):.1f})')

axes[1].hist(n_dup, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Duplications')
axes[1].set_title(f'Duplications (mean={sum(n_dup)/len(n_dup):.1f})')

axes[2].hist(n_transfer, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('Transfers')
axes[2].set_title(f'Transfers (mean={sum(n_transfer)/len(n_transfer):.1f})')

axes[3].hist(n_loss, bins=30, edgecolor='black', alpha=0.7, color='red')
axes[3].set_xlabel('Losses')
axes[3].set_title(f'Losses (mean={sum(n_loss)/len(n_loss):.1f})')

plt.tight_layout()
plt.show()